# Типология клиентской базы — признаки

Собираю признаки для кластеризации: смотрю какие таблицы использовать, отбрасываю мусорные, считаю признаки в индивидуальном окне наблюдения.

Ниже пока sample по операциям (3 клиента), пока не догрузилась полная таблица — логика та же.

## Окружение

Python 3.9, conda `churn_model`.
```python
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn psycopg2-binary
```
После установки — Kernel → Restart. Подключаюсь через psycopg2 напрямую (sqlalchemy конфликтует с typing_extensions на 3.9).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import psycopg2

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

DATA_DIR = Path('data')  # положите сюда все seg_*.csv выгрузки из Postgres
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 0. Подключение к Postgres

Тащу таблицы из academy_db. CSV ниже — фолбэк, если БД недоступна.

In [ ]:
# Set these environment variables locally before running the notebook.
# Never commit database credentials to GitHub.
import os
PG_LOGIN = os.getenv("PG_LOGIN")
PG_PASSWORD = os.getenv("PG_PASSWORD")
if not PG_LOGIN or not PG_PASSWORD:
    raise EnvironmentError(
        "Set PG_LOGIN and PG_PASSWORD environment variables before connecting."
    )
PG_HOST = "thomas.proxy.rlwy.net"
PG_PORT = 51432
PG_DB = "academy_db"

try:
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT, dbname=PG_DB,
        user=PG_LOGIN, password=PG_PASSWORD,
    )
    check_tables_sql = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public' AND table_name LIKE 'seg_%'
        ORDER BY table_name;
    """
    with conn.cursor() as cur:
        cur.execute(check_tables_sql)
        seg_tables_in_db = [row[0] for row in cur.fetchall()]
    DB_AVAILABLE = True
    print("Подключение успешно. Таблицы с префиксом seg_ в public:")
    for t in seg_tables_in_db:
        print(" -", t)
except Exception as e:
    DB_AVAILABLE = False
    seg_tables_in_db = []
    conn = None
    print("Не удалось подключиться к Postgres — ниже сработает CSV-фолбэк.")
    print("Ошибка:", e)


In [ ]:
sql_table_map = {
    'customers':       'seg_customers',
    'customers_360':   'seg_customers_360',
    'balances':        'seg_balances',
    'revenue_monthly': 'seg_revenue_monthly',
    'products':        'seg_products',
    'tariff_current':  'seg_tariff_current',
    'rfm_old':         'seg_rfm_old',
    'temp':            'seg_temp',
    'operations':      'seg_transactions',
    'transactions_archive': 'seg_transactions_archive',  # подозрительная — проверяем ниже, не используем как есть
}

import warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

class _ProgressFile:
    def __init__(self, path, print_every_bytes=20_000_000):
        self.f = open(path, 'wb')
        self.bytes_written = 0
        self.print_every = print_every_bytes
        self._last_print = 0
    def write(self, data):
        if isinstance(data, str):
            data = data.encode('utf-8')
        self.f.write(data)
        self.bytes_written += len(data)
        if self.bytes_written - self._last_print >= self.print_every:
            print('  MB streamed: ' + format(self.bytes_written // 1_000_000, ',') + chr(13), end='')
            self._last_print = self.bytes_written
    def close(self):
        self.f.close()

dfs = {}

if DB_AVAILABLE:
    for key, table_name in sql_table_map.items():
        if table_name not in seg_tables_in_db:
            print(f"[!] {table_name} не найдена в БД, пропускаю")
            continue

        if key == 'operations':
            with conn.cursor() as _cur:
                _cur.execute(f"SELECT MAX(txn_ts) FROM public.{table_name}")
                max_ts = _cur.fetchone()[0]
            cutoff_months = 8
            select_sql = (
                f"COPY (SELECT * FROM public.{table_name} "
                f"WHERE txn_ts >= TIMESTAMP '{max_ts}' - INTERVAL '{cutoff_months} months') "
                f"TO STDOUT WITH (FORMAT CSV, HEADER TRUE)"
            )
            print(f'  {table_name}: тяну только последние {cutoff_months} мес. (макс. дата: {max_ts})')
        elif key == 'transactions_archive':
            select_sql = f"COPY (SELECT * FROM public.{table_name}) TO STDOUT WITH (FORMAT CSV, HEADER TRUE)"
        else:
            select_sql = None  # обычные таблицы — через pd.read_sql ниже, без стриминга

        if select_sql is not None:
            raw_path = DATA_DIR / f'{table_name}_raw.csv'
            _pf = _ProgressFile(raw_path)
            with conn.cursor() as cur:
                cur.copy_expert(select_sql, _pf)
            _pf.close()
            print()
            print(f'  {table_name}: скачано {_pf.bytes_written/1_000_000:.1f} MB, читаю в pandas...')
            dfs[key] = pd.read_csv(raw_path)
        else:
            dfs[key] = pd.read_sql(f'SELECT * FROM public.{table_name}', conn)

        print(f"{key:16s} <- {table_name:20s} shape={dfs[key].shape}")

    if 'operations' in dfs:
        dfs['operations']['txn_ts'] = pd.to_datetime(dfs['operations']['txn_ts'])

    conn.close()

    for key, df in dfs.items():
        df.to_csv(DATA_DIR / f'{key}_from_sql.csv', index=False)
else:
    print("БД недоступна в этом запуске — используются CSV-выгрузки из раздела 1 (фолбэк).")

## 1. CSV-фолбэк

Если раздел 0 подключился, этот блок пропускается.

In [ ]:
if not DB_AVAILABLE:
    tables = {
        'customers':        'seg_customers_202608190209.csv',
        'customers_360':    'seg_customers_360_202608190211.csv',
        'balances':         'seg_balances_202608190206.csv',
        'revenue_monthly':  'seg_revenue_monthly_202608190217.csv',
        'products':         'seg_products_202608190213.csv',
        'tariff_current':   'seg_tariff_current_202608190226.csv',
        'rfm_old':          'seg_rfm_old_202608190222.csv',
        'temp':             'seg_temp_202608190227.csv',
    }

    dfs = {}
    for name, fname in tables.items():
        path = DATA_DIR / fname
        if path.exists():
            dfs[name] = pd.read_csv(path)
        else:
            print(f'[!] {fname} не найден — положите файл в data/')

    for name, df in dfs.items():
        print(f"{name:16s} shape={df.shape}  cols={list(df.columns)}")

else:
    print('DB_AVAILABLE=True — секция CSV пропущена, dfs уже заполнен через SQL (раздел 0).')

In [ ]:
if 'operations' in dfs:
    ops = dfs['operations']
    print('operations уже загружены через SQL (раздел 0):', ops.shape,
          f"({ops['customer_id'].nunique()} уникальных клиентов)")
else:
    if DB_AVAILABLE:
        expected_table = sql_table_map.get("operations", "?")
        print(f"[!] Подключение к БД есть, но '{expected_table}' не нашлась среди таблиц public.seg_%, или ячейка с SQL-извлечением не выполнялась в этой сессии ядра:")
        print("    Найденные таблицы:", seg_tables_in_db)
        print("    Проверьте точное имя таблицы в БД (регистр, префикс) и права доступа для вашего логина.")

    ops_path = DATA_DIR / 'seg_transactions.csv'
    ops_sample_path = DATA_DIR / 'seg_operations_sample.csv'

    if ops_path.exists():
        ops = pd.read_csv(ops_path, parse_dates=['txn_ts'])
        print('Загружен ПОЛНЫЙ файл операций (CSV):', ops.shape)
    elif ops_sample_path.exists():
        ops = pd.read_csv(ops_sample_path, parse_dates=['txn_ts'])
        print('[!] Работаю на CSV-sample:', ops.shape,
              f"({ops['customer_id'].nunique()} клиентов вместо 20000)")
    else:
        raise FileNotFoundError(
            "Нет данных по операциям ни из SQL, ни из CSV. "
            f"Положите файл в {ops_path} (полный) или {ops_sample_path} (sample), "
            "либо разберитесь, почему 'seg_operations' недоступна в БД (см. подсказку выше)."
        )

    dfs['operations'] = ops

ops.head()


## 2. Что отбрасываем

В выгрузке есть лишнее — проверяю каждую подозрительную таблицу отдельно.

In [ ]:
print('seg_temp:')
print(dfs['temp'])
print('-> пустой файл (0 строк данных), явный мусор. Не используется.')


In [ ]:
rfm = dfs['rfm_old']
cust = dfs['customers']
print('seg_rfm_old:', rfm.shape, '| уникальных клиентов:', rfm['customer_id'].nunique())
print('seg_customers:', cust.shape[0], 'клиентов всего')
print('Доля клиентов БЕЗ записи в rfm_old:',
      round(1 - rfm['customer_id'].nunique() / cust['customer_id'].nunique(), 3))
rfm.describe()


`seg_rfm_old` не покрывает всю базу, окно расчёта не задокументировано. Использую только для сравнения, не как признак.

In [ ]:
c360 = dfs['customers_360']
print(c360['crm_segment'].value_counts())
print()
print('Корреляция turnover_total (360) с суммой avg_balance (balances) — быстрая проверка согласованности:')
bal_sum = dfs['balances'].groupby('customer_id')['avg_balance_uzs'].sum()
merged = c360.set_index('customer_id')[['turnover_total']].join(bal_sum, how='inner')
print(merged.corr())


`crm_segment` в `seg_customers_360` — готовый сегмент по обороту за весь период, без поправки на разный стаж клиентов. Беру как бенчмарк для сравнения, не как признак.

In [ ]:
dfs['tariff_current']


In [ ]:
if 'transactions_archive' in dfs and 'operations' in dfs:
    arch = dfs['transactions_archive']
    real = dfs['operations']
    print('seg_transactions          :', real.shape, '| customer_id nunique:', real['customer_id'].nunique())
    print('seg_transactions_archive  :', arch.shape, '| customer_id nunique:', arch['customer_id'].nunique())
    if 'txn_ts' in arch.columns:
        arch_ts = pd.to_datetime(arch['txn_ts'])
        print('archive date range:', arch_ts.min(), '->', arch_ts.max())
    real_ts = pd.to_datetime(real['txn_ts'])
    print('real date range    :', real_ts.min(), '->', real_ts.max())
    overlap = set(arch['txn_id']) & set(real['txn_id']) if 'txn_id' in arch.columns else set()
    print('пересечение txn_id с боевой таблицей:', len(overlap))


`seg_transactions_archive` — служебная копия, признаки из неё не считаю.

## 3. Окно наблюдения

Клиенты пришли в разное время — если считать признаки за весь период, типология получится по стажу, а не по поведению.

In [ ]:
cust['signup_date'] = pd.to_datetime(cust['signup_date'])
tenure_days = (pd.Timestamp('2025-12-31') - cust['signup_date']).dt.days
print(tenure_days.describe())
print()
print('Доля клиентов со стажем < 6 месяцев (180 дней):',
      round((tenure_days < 180).mean(), 3))
print('Доля клиентов со стажем < 3 месяцев (90 дней):',
      round((tenure_days < 90).mean(), 3))


In [ ]:
bal = dfs['balances'].copy()
bal['month_start'] = pd.to_datetime(bal['month_start'])

cv_per_customer = (
    bal.groupby('customer_id')['avg_balance_uzs']
       .agg(['mean', 'std'])
       .assign(cv=lambda d: d['std'] / d['mean'].replace(0, np.nan))
)
print(cv_per_customer['cv'].describe())
print()
print('Доля клиентов с CV остатка > 1 (т.е. разброс больше среднего — поведение "плывёт"):',
      round((cv_per_customer['cv'] > 1).mean(), 3))


Часть клиентов со стажем < 6 месяцев, у части CV остатка > 1.

Беру окно 6 месяцев от последней активности каждого клиента, а не от календаря — иначе клиенты, пришедшие раньше и позже, окажутся в разных фазах.

In [ ]:
WINDOW_MONTHS = 6

last_month_per_cust = bal.groupby('customer_id')['month_start'].max().rename('window_end')
window_start = (last_month_per_cust - pd.DateOffset(months=WINDOW_MONTHS - 1))

window_df = pd.DataFrame({'window_end': last_month_per_cust, 'window_start': window_start})
window_df['months_available'] = (
    bal.groupby('customer_id')['month_start'].nunique()
)
print(window_df.head())
print()
print('Медиана доступных месяцев истории:', window_df['months_available'].median())
print('Доля клиентов с < 6 месяцами истории (окно будет короче стандартного):',
      round((window_df['months_available'] < WINDOW_MONTHS).mean(), 3))


## 4. Оси

Беру 5 осей вместо одной:

| Ось | Вопрос | Источник |
|---|---|---|
| Денежный объём | Сколько денег держит и насколько стабильно | `seg_balances` |
| Доходность | Сколько банк зарабатывает и за счёт чего | `seg_revenue_monthly` |
| Поведение | Как тратит — каналы, категории | `seg_transactions` |
| Стадия жизни | Стаж, вовлечённость | `seg_customers` + `seg_products` |
| Демография | Возраст, пол | `seg_customers` |

Не беру как признаки: `crm_segment` и `seg_rfm_old` (только для сравнения), `region` (13 категорий, one-hot забьёт остальные признаки — смотрю после кластеризации), `full_name`, `tariff_code` (не вариативен).

Пол сначала оставляла вместе с возрастом, но `is_male` (дисбаланс 66/34) задавил геометрию — 5 из 8 кластеров вышли на 84-100% одного пола. Убрала из кластеризации, оставила только в разрезе (раздел 16).

## 5. Признаки

Считаю блок за блоком, в пределах окна из раздела 3.

In [ ]:
def clip_to_window(df, ts_col, cid_col='customer_id'):
    '''Обрезает датафрейм по индивидуальному окну наблюдения каждого клиента.'''
    merged = df.merge(window_df, left_on=cid_col, right_index=True, how='inner')
    mask = (merged[ts_col] >= merged['window_start']) & (merged[ts_col] <= merged['window_end'])
    return merged.loc[mask].drop(columns=['window_start', 'window_end', 'months_available'])


### 5.1 Денежный блок

Средний остаток, CV (стабильность), тренд.

In [ ]:
if 'WINDOW_MONTHS' not in dir():
    WINDOW_MONTHS = 6

if 'bal' not in dir():
    bal = dfs['balances'].copy()
    bal['month_start'] = pd.to_datetime(bal['month_start'])

if 'window_df' not in dir():
    last_month_per_cust = bal.groupby('customer_id')['month_start'].max().rename('window_end')
    window_start = (last_month_per_cust - pd.DateOffset(months=WINDOW_MONTHS - 1))
    window_df = pd.DataFrame({'window_end': last_month_per_cust, 'window_start': window_start})
    window_df['months_available'] = bal.groupby('customer_id')['month_start'].nunique()

if 'clip_to_window' not in dir():
    def clip_to_window(df, ts_col, cid_col='customer_id'):
        merged = df.merge(window_df, left_on=cid_col, right_index=True, how='inner')
        mask = (merged[ts_col] >= merged['window_start']) & (merged[ts_col] <= merged['window_end'])
        return merged.loc[mask].drop(columns=['window_start', 'window_end', 'months_available'])

print('Готово: WINDOW_MONTHS, bal, window_df, clip_to_window — все на месте.')

bal_w = clip_to_window(bal, 'month_start')

bal_features = bal_w.groupby('customer_id')['avg_balance_uzs'].agg(
    balance_mean_w='mean',
    balance_std_w='std',
).reset_index()
bal_features['balance_cv_w'] = (
    bal_features['balance_std_w'] / bal_features['balance_mean_w'].replace(0, np.nan)
).fillna(0)

def _trend(g):
    g = g.sort_values('month_start')
    if len(g) < 2:
        return 0.0
    x = np.arange(len(g))
    y = g['avg_balance_uzs'].values
    return np.polyfit(x, y, 1)[0]

trend = bal_w.groupby('customer_id').apply(_trend).rename('balance_trend_w')
bal_features = bal_features.merge(trend, on='customer_id')
bal_features.head()

### 5.2 Доходность

Не только сумма, но структура — эквайринг vs float.

In [ ]:
if 'WINDOW_MONTHS' not in dir():
    WINDOW_MONTHS = 6

if 'bal' not in dir():
    bal = dfs['balances'].copy()
    bal['month_start'] = pd.to_datetime(bal['month_start'])

if 'window_df' not in dir():
    last_month_per_cust = bal.groupby('customer_id')['month_start'].max().rename('window_end')
    window_start = (last_month_per_cust - pd.DateOffset(months=WINDOW_MONTHS - 1))
    window_df = pd.DataFrame({'window_end': last_month_per_cust, 'window_start': window_start})
    window_df['months_available'] = bal.groupby('customer_id')['month_start'].nunique()

if 'clip_to_window' not in dir():
    def clip_to_window(df, ts_col, cid_col='customer_id'):
        merged = df.merge(window_df, left_on=cid_col, right_index=True, how='inner')
        mask = (merged[ts_col] >= merged['window_start']) & (merged[ts_col] <= merged['window_end'])
        return merged.loc[mask].drop(columns=['window_start', 'window_end', 'months_available'])

print('Готово: WINDOW_MONTHS, bal, window_df, clip_to_window — все на месте.')

rev = dfs['revenue_monthly'].copy()
rev['month_start'] = pd.to_datetime(rev['month_start'])
rev_w = clip_to_window(rev, 'month_start')

rev_features = rev_w.groupby('customer_id').agg(
    net_revenue_mean_w=('net_revenue_uzs', 'mean'),
    net_revenue_sum_w=('net_revenue_uzs', 'sum'),
    interchange_sum_w=('interchange_uzs', 'sum'),
    float_income_sum_w=('float_income_uzs', 'sum'),
    transfer_fee_sum_w=('transfer_fee_uzs', 'sum'),
    atm_cost_sum_w=('atm_cost_uzs', 'sum'),
).reset_index()

rev_components = rev_features['interchange_sum_w'] + rev_features['float_income_sum_w'] + rev_features['transfer_fee_sum_w']
rev_features['interchange_share_w'] = (rev_features['interchange_sum_w'] / rev_components.replace(0, np.nan)).fillna(0)
rev_features['float_share_w'] = (rev_features['float_income_sum_w'] / rev_components.replace(0, np.nan)).fillna(0)
rev_features.head()

### 5.3 Поведенческий блок

Доли по каналам, энтропия категорий, частота, средний чек.

Пока на sample, полная таблица логику не меняет.

In [ ]:
ops_w = clip_to_window(ops, 'txn_ts')

out_ops = ops_w[ops_w['direction'] == 'out']

channel_counts = out_ops.groupby(['customer_id', 'channel']).size().unstack(fill_value=0)
channel_shares = channel_counts.div(channel_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
channel_shares.columns = [f'channel_share_{c}' for c in channel_shares.columns]

atm_share = channel_shares.get('channel_share_atm', pd.Series(dtype=float)).rename('atm_share_of_out_txns')

def _entropy(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return -(p * np.log(p)).sum()

mcc_entropy = (
    out_ops.groupby(['customer_id', 'mcc_group']).size()
    .groupby('customer_id').apply(_entropy)
    .rename('mcc_category_entropy_w')
)

txn_freq = ops_w.groupby('customer_id').size().rename('txn_count_w')
avg_ticket = out_ops.groupby('customer_id')['amount_uzs'].mean().rename('avg_out_ticket_w')

p2p_in = ops_w[(ops_w['channel'] == 'p2p') & (ops_w['direction'] == 'in')].groupby('customer_id')['amount_uzs'].sum()
total_out = out_ops.groupby('customer_id')['amount_uzs'].sum()
p2p_in_ratio = (p2p_in / total_out.replace(0, np.nan)).fillna(0).rename('p2p_in_to_spend_ratio_w')

behavior_features = (
    channel_shares
    .join(mcc_entropy, how='left')
    .join(txn_freq, how='left')
    .join(avg_ticket, how='left')
    .join(p2p_in_ratio, how='left')
    .fillna(0)
    .reset_index()
)
behavior_features.head()


### 5.4 Жизненный цикл и демография

Стаж, число продуктов, возраст, пол.

In [ ]:
cust_features = cust[['customer_id', 'signup_date']].copy()
cust_features = cust_features.merge(window_df[['window_end']], left_on='customer_id', right_index=True, how='left')
cust_features['tenure_months_at_window_end'] = (
    (cust_features['window_end'] - cust_features['signup_date']).dt.days / 30.44
).round(1)
cust_features = cust_features.drop(columns=['window_end'])

demo = cust[['customer_id', 'birth_date', 'gender', 'region']].copy()
demo['birth_date'] = pd.to_datetime(demo['birth_date'])
demo = demo.merge(window_df[['window_end']], left_on='customer_id', right_index=True, how='left')
demo['age_years'] = ((demo['window_end'] - demo['birth_date']).dt.days / 365.25).round(1)
demo['is_male'] = (demo['gender'] == 'M').astype(int)
region_lookup = demo[['customer_id', 'region', 'gender']].set_index('customer_id')

cust_features = cust_features.merge(demo[['customer_id', 'age_years', 'is_male']], on='customer_id', how='left')

prod = dfs['products'].copy()
n_products = prod.groupby('customer_id')['product_code'].nunique().rename('n_products')
has_credit_line = (
    prod.assign(flag=prod['product_code'] == 'credit_line')
        .groupby('customer_id')['flag'].max().rename('has_credit_line')
)

lifecycle_features = (
    cust_features.set_index('customer_id')
    .join(n_products, how='left')
    .join(has_credit_line, how='left')
    .fillna({'n_products': 0, 'has_credit_line': False, 'age_years': cust_features['age_years'].median(), 'is_male': 0})
    .reset_index()
)
lifecycle_features.head()

## 6. Сборка таблицы признаков

Собираю блоки по customer_id. Клиенты без операций получают 0 — не пропуск данных, а валидный тип ("спящий").

In [ ]:
feature_table = (
    lifecycle_features
    .merge(bal_features, on='customer_id', how='left')
    .merge(rev_features, on='customer_id', how='left')
    .merge(behavior_features, on='customer_id', how='left')
)

numeric_cols = feature_table.select_dtypes(include=[np.number]).columns
feature_table[numeric_cols] = feature_table[numeric_cols].fillna(0)

print('Итоговая таблица признаков:', feature_table.shape)
feature_table.head()


## 7. Сводка

| Признак | Источник | Зачем |
|---|---|---|
| `tenure_months_at_window_end` | `customers` | стадия жизни |
| `age_years`, `is_male` | `customers` | демография |
| `n_products`, `has_credit_line` | `products` | вовлечённость |
| `balance_mean_w`, `balance_cv_w`, `balance_trend_w` | `balances` | объём и стабильность денег |
| `net_revenue_*_w`, `interchange_share_w`, `float_share_w` | `revenue_monthly` | доходность и её структура |
| `channel_share_*`, `atm_share_of_out_txns` | `operations` | канал |
| `mcc_category_entropy_w` | `operations` | широта трат |
| `txn_count_w`, `avg_out_ticket_w` | `operations` | активность |
| `p2p_in_to_spend_ratio_w` | `operations` | зависимость от переводов |

Не используются: `seg_rfm_old`, `crm_segment`/`turnover_total` (бенчмарк), демография как отдельные поля, `tariff_code`, `seg_temp`.

## 8. Масштабирование

Несколько клиентов с огромными оборотами могут утащить центры кластеров на себя. Winsorize (1-99 перцентиль) + StandardScaler — обрезка ограничивает влияние выбросов, не удаляет клиентов.

In [ ]:
from sklearn.preprocessing import StandardScaler

EXCLUDE_FROM_CLUSTERING = {'is_male'}
FEATURE_COLS = [c for c in feature_table.columns
                if c != 'customer_id' and c not in EXCLUDE_FROM_CLUSTERING
                and pd.api.types.is_numeric_dtype(feature_table[c])]
print('Признаков в матрице:', len(FEATURE_COLS))
print(FEATURE_COLS)

X_raw = feature_table[FEATURE_COLS].astype(float).copy()

# winsorize: обрезаем по 1/99 перцентилю каждый столбец отдельно
X_wins = X_raw.copy()
clip_bounds = {}
for col in FEATURE_COLS:
    lo, hi = X_raw[col].quantile(0.01), X_raw[col].quantile(0.99)
    clip_bounds[col] = (lo, hi)
    X_wins[col] = X_raw[col].clip(lo, hi)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_wins), columns=FEATURE_COLS, index=feature_table.index)
X_scaled.insert(0, 'customer_id', feature_table['customer_id'].values)
print('Матрица признаков готова:', X_scaled.shape)
X_scaled.head()

## 9. Число кластеров

Silhouette — ориентир, не финальный аргумент. Число защищается тем, что каждый кластер имеет свой интерпретируемый портрет (раздел 11).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_mat = X_scaled[FEATURE_COLS].values
n_samples = X_mat.shape[0]

k_range = [k for k in range(2, 13) if k < n_samples]  # шире диапазон — преподаватель просил более гранулярные типы
if not k_range:
    print(f'[!] Слишком мало клиентов в текущих данных (n={n_samples}) для перебора k — '
          f'это ожидаемо на sample-данных, на полной БД (20000 клиентов) отработает нормально.')
else:
    results = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_mat)
        sil = silhouette_score(X_mat, labels) if len(set(labels)) > 1 else float('nan')
        results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil})
    k_table = pd.DataFrame(results)
    print(k_table)


Не беру k с максимальным silhouette автоматически — смотрю на профили кластеров для 2-3 кандидатов и выбираю то k, где каждый кластер описывается разным поведением. K_FINAL фиксирую ниже.

In [ ]:
# Итоговое число кластеров — подставьте после анализа таблицы выше и профилей ниже.
K_FINAL = min(8, max(2, n_samples - 1))  # безопасный дефолт для маленького sample; на полных данных выберите осознанно из таблицы выше — цель раздела 9, обычно 6-10 для более детальной типологии
print('K_FINAL =', K_FINAL)


## 10. Кластеризация: K-means и DBSCAN

Строю оба метода — если дают похожую структуру, это дополнительный аргумент устойчивости.

In [ ]:
kmeans_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
feature_table['cluster_kmeans'] = kmeans_final.fit_predict(X_mat)
print(feature_table['cluster_kmeans'].value_counts().sort_index())


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

min_samples = max(3, 2 * len(FEATURE_COLS))
if n_samples > min_samples:
    nn = NearestNeighbors(n_neighbors=min_samples).fit(X_mat)
    distances, _ = nn.kneighbors(X_mat)
    k_dist = np.sort(distances[:, -1])
    eps_guess = float(np.percentile(k_dist, 90))
    dbscan = DBSCAN(eps=eps_guess, min_samples=min_samples)
    feature_table['cluster_dbscan'] = dbscan.fit_predict(X_mat)
    print('eps =', round(eps_guess, 3), '| min_samples =', min_samples)
    print(feature_table['cluster_dbscan'].value_counts().sort_index())
    print('(-1 = шум/выбросы по мнению DBSCAN — отдельная от кластеров категория)')
else:
    print(f'[!] n_samples={n_samples} слишком мало относительно min_samples={min_samples} — на sample не показательно, на полных данных отработает.')
    feature_table['cluster_dbscan'] = -1


In [ ]:
from sklearn.metrics import adjusted_rand_score

if feature_table['cluster_dbscan'].nunique() > 1:
    ari_km_db = adjusted_rand_score(feature_table['cluster_kmeans'], feature_table['cluster_dbscan'])
    print('ARI между K-means и DBSCAN (без учёта шумовых точек DBSCAN):', round(ari_km_db, 3))
else:
    print('DBSCAN нашёл один сегмент/шум целиком — сравнение с K-means неинформативно на этих данных.')


## 11. Портрет каждого кластера

Средние значения в исходном масштабе — так портрет читается человеком.

In [ ]:
profile_cols = ['tenure_months_at_window_end', 'age_years', 'is_male', 'n_products', 'balance_mean_w', 'balance_cv_w',
                'net_revenue_sum_w', 'channel_share_atm', 'channel_share_ecom', 'channel_share_pos',
                'channel_share_p2p', 'mcc_category_entropy_w', 'txn_count_w', 'avg_out_ticket_w',
                'p2p_in_to_spend_ratio_w']
profile_cols = [c for c in profile_cols if c in feature_table.columns]

cluster_profile = feature_table.groupby('cluster_kmeans')[profile_cols].mean().round(2)
cluster_sizes = feature_table.groupby('cluster_kmeans').size().rename('n_customers')
cluster_profile = cluster_sizes.to_frame().join(cluster_profile)
cluster_profile


## 12. Сравнение с квартилями оборота

Делю базу на 4 части по обороту без кластеризации, сравниваю с типологией не только по ARI, но и содержательно.

In [ ]:
turnover_col = 'avg_out_ticket_w' if 'avg_out_ticket_w' in feature_table.columns else 'net_revenue_sum_w'
turnover_proxy = (feature_table['txn_count_w'] * feature_table['avg_out_ticket_w']) if \
    {'txn_count_w', 'avg_out_ticket_w'}.issubset(feature_table.columns) else feature_table[turnover_col]

feature_table['turnover_proxy'] = turnover_proxy
feature_table['quartile'] = pd.qcut(feature_table['turnover_proxy'].rank(method='first'), 4,
                                     labels=['Q1_low', 'Q2', 'Q3', 'Q4_high'])

ari_quartile = adjusted_rand_score(feature_table['cluster_kmeans'], feature_table['quartile'].cat.codes)
print('ARI между нашей типологией и квартилями оборота:', round(ari_quartile, 3))
print('(низкое значение = типологии структурно разные — квартиль НЕ повторяет то, что делает кластеризация)')


In [ ]:
cross = pd.crosstab(feature_table['quartile'], feature_table['cluster_kmeans'])
print(cross)
print()
n_mixed_quartiles = (cross.astype(bool).sum(axis=1) > 1).sum()
print(f'Квартилей, где смешано больше одного нашего кластера: {n_mixed_quartiles} из {cross.shape[0]}')
print('Если это число > 0 — значит внутри одной "денежной" группы по обороту реально есть')
print('поведенчески разные клиенты, которых квартиль не различает, а наша типология — различает.')


## 13. Устойчивость

4 проверки: другой random_state, половина выборки, без последнего месяца, изменённый набор признаков. Сравниваю через ARI на пересечении клиентов.

In [ ]:
stability_results = []

# (a) другой random_state
km_alt_seed = KMeans(n_clusters=K_FINAL, random_state=123, n_init=10).fit_predict(X_mat)
stability_results.append({
    'проверка': 'другой random_state',
    'ARI': round(adjusted_rand_score(feature_table['cluster_kmeans'], km_alt_seed), 3)
})

# (b) половина выборки
rng = np.random.RandomState(42)
half_idx = rng.choice(n_samples, size=max(2, n_samples // 2), replace=False)
if len(half_idx) >= K_FINAL:
    km_half = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit_predict(X_mat[half_idx])
    ari_half = adjusted_rand_score(feature_table['cluster_kmeans'].values[half_idx], km_half)
    stability_results.append({'проверка': 'половина выборки', 'ARI': round(ari_half, 3)})
else:
    stability_results.append({'проверка': 'половина выборки', 'ARI': float('nan')})

# (c) изменённый набор признаков — убираем одну "ось" целиком (например, поведенческий блок)
alt_cols = [c for c in FEATURE_COLS if not c.startswith('channel_share_') and c != 'mcc_category_entropy_w']
if len(alt_cols) >= 2:
    X_alt = X_scaled[alt_cols].values
    km_alt_feat = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit_predict(X_alt)
    ari_alt_feat = adjusted_rand_score(feature_table['cluster_kmeans'], km_alt_feat)
    stability_results.append({'проверка': 'без канальных/категорийных признаков', 'ARI': round(ari_alt_feat, 3)})

stability_table = pd.DataFrame(stability_results)
stability_table


Проверка «без последнего месяца» пересчитывает признаки заново на укороченном окне, не просто вырезает строки.

In [ ]:
window_df_minus1 = window_df.copy()
window_df_minus1['window_end'] = window_df_minus1['window_end'] - pd.DateOffset(months=1)
window_df_minus1['window_start'] = window_df_minus1['window_end'] - pd.DateOffset(months=WINDOW_MONTHS - 1)

def clip_to_window_alt(df, ts_col, wdf, cid_col='customer_id'):
    merged = df.merge(wdf, left_on=cid_col, right_index=True, how='inner')
    mask = (merged[ts_col] >= merged['window_start']) & (merged[ts_col] <= merged['window_end'])
    return merged.loc[mask].drop(columns=['window_start', 'window_end'], errors='ignore')

bal_w_m1 = clip_to_window_alt(bal, 'month_start', window_df_minus1[['window_start', 'window_end']])
bal_feat_m1 = bal_w_m1.groupby('customer_id')['avg_balance_uzs'].agg(
    balance_mean_w='mean', balance_std_w='std'
).reset_index()
bal_feat_m1['balance_cv_w'] = (bal_feat_m1['balance_std_w'] / bal_feat_m1['balance_mean_w'].replace(0, np.nan)).fillna(0)

common_ids = set(bal_feat_m1['customer_id']) & set(feature_table['customer_id'])
if len(common_ids) >= K_FINAL:
    check_df = feature_table[feature_table['customer_id'].isin(common_ids)].copy()
    bal_feat_m1_idx = bal_feat_m1.set_index('customer_id')
    for col in ['balance_mean_w', 'balance_std_w', 'balance_cv_w']:
        if col in check_df.columns:
            check_df = check_df.set_index('customer_id')
            check_df[col] = bal_feat_m1_idx[col].reindex(check_df.index).fillna(check_df[col])
            check_df = check_df.reset_index()

    if len(check_df) >= K_FINAL:
        X_check_wins = check_df[FEATURE_COLS].copy()
        for col in FEATURE_COLS:
            lo, hi = clip_bounds[col]
            X_check_wins[col] = X_check_wins[col].clip(lo, hi)
        X_m1 = StandardScaler().fit_transform(X_check_wins.fillna(0))
        km_m1 = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit_predict(X_m1)
        ari_m1 = adjusted_rand_score(check_df['cluster_kmeans'], km_m1)
        stability_table = pd.concat([stability_table, pd.DataFrame([
            {'проверка': 'без последнего месяца (полное признаковое пространство)', 'ARI': round(ari_m1, 3)}
        ])], ignore_index=True)

stability_table


## 14. Правило для нового клиента

Ближайший центр кластера по евклидову расстоянию. Проверяю на отложенной выборке (train/test).

In [ ]:
from sklearn.model_selection import train_test_split

if n_samples >= 2 * K_FINAL:
    idx_all = np.arange(n_samples)
    idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=42)

    km_train = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit(X_mat[idx_train])

    test_labels_by_rule = km_train.predict(X_mat[idx_test])

    km_full_for_check = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10).fit_predict(X_mat)
    test_labels_full = km_full_for_check[idx_test]

    ari_newclient_rule = adjusted_rand_score(test_labels_full, test_labels_by_rule)
    print(f'Отложено {len(idx_test)} клиентов ({len(idx_test)/n_samples:.0%}) — не участвовали в обучении центров.')
    print('ARI между "правило ближайшего центра" и "если бы клиент был в обучении с самого начала":',
          round(ari_newclient_rule, 3))
    print('Высокое значение = правило надёжно переносится на новых клиентов, не только на обучающих.')
else:
    print(f'[!] n_samples={n_samples} мало для honest train/test на этих данных — код рабочий, '
          f'на полной БД (20000 клиентов, test 20% ≈ 4000) даст содержательный результат.')


## 15. Названия и деньги

Названия не на глаз — генерируются по z-score отклонению от среднего по базе.

In [ ]:
NAME_RULES = [
    ('channel_share_atm',      lambda z: z > 0.5,  'наличный тип: живёт через банкоматы'),
    ('channel_share_ecom',     lambda z: z > 0.5,  'онлайн-покупатель: основной канал — интернет-эквайринг'),
    ('p2p_in_to_spend_ratio_w',lambda z: z > 0.5,  'живёт на переводах: доходы преимущественно через P2P'),
    ('balance_cv_w',           lambda z: z > 0.5,  'нестабильный остаток: баланс сильно скачет по месяцам'),
    ('balance_mean_w',         lambda z: z > 0.7,  'высокий постоянный остаток: держит деньги на счету'),
    ('balance_mean_w',         lambda z: z < -0.5, 'низкий остаток: почти не хранит деньги на счету'),
    ('net_revenue_sum_w',      lambda z: z > 0.7,  'высокодоходный: банк много зарабатывает на этом клиенте'),
    ('net_revenue_sum_w',      lambda z: z < -0.5, 'низкодоходный: банк почти не зарабатывает на этом клиенте'),
    ('interchange_share_w',    lambda z: z > 0.5,  'доход через эквайринг: платит картой в торговых точках'),
    ('float_share_w',          lambda z: z > 0.5,  'доход через float: банк зарабатывает на остатках'),
    ('tenure_months_at_window_end', lambda z: z < -0.5, 'новый клиент: короткая история в банке'),
    ('tenure_months_at_window_end', lambda z: z > 0.7,  'старожил: давняя история в банке'),
    ('n_products',             lambda z: z > 0.5,  'вовлечён в продукты: пользуется несколькими продуктами банка'),
    ('mcc_category_entropy_w', lambda z: z > 0.5,  'широкий профиль трат: тратит на разные категории'),
    ('txn_count_w',            lambda z: z < -0.5, 'низкая активность: мало операций за период'),
    ('txn_count_w',            lambda z: z > 0.7,  'высокая активность: много операций за период'),
    ('age_years',              lambda z: z < -0.6, 'молодёжный сегмент: заметно моложе средней базы'),
    ('age_years',              lambda z: z > 0.6,  'возрастной сегмент: заметно старше средней базы'),
    ('is_male',                lambda z: z > 0.5,  'преимущественно мужской состав'),
    ('is_male',                lambda z: z < -0.5, 'преимущественно женский состав'),
]

cluster_centers_z = feature_table.groupby('cluster_kmeans')[FEATURE_COLS].mean()
overall_mean = feature_table[FEATURE_COLS].mean()
overall_std = feature_table[FEATURE_COLS].std().replace(0, 1)
cluster_z = (cluster_centers_z - overall_mean) / overall_std

def auto_name(cluster_id):
    z_row = cluster_z.loc[cluster_id]
    tags = [desc for col, rule, desc in NAME_RULES if col in z_row.index and rule(z_row[col])]
    return '; '.join(tags) if tags else 'смешанный / нейтральный профиль'

cards = []
for cl in sorted(feature_table['cluster_kmeans'].unique()):
    sub = feature_table[feature_table['cluster_kmeans'] == cl]
    cards.append({
        'cluster': cl,
        'название (авто)': auto_name(cl),
        'размер': len(sub),
        'доля базы': round(len(sub) / len(feature_table), 3),
        'доход банку сейчас (сумма net_revenue_sum_w)': round(sub['net_revenue_sum_w'].sum(), 0) if 'net_revenue_sum_w' in sub.columns else None,
        'средний доход на клиента': round(sub['net_revenue_sum_w'].mean(), 0) if 'net_revenue_sum_w' in sub.columns else None,
    })

cluster_cards = pd.DataFrame(cards)
cluster_cards


Что предлагаем по типовым тегам:
- наличный тип → кэшбэк за безнал
- онлайн-покупатель → кэшбэк в ecom-категориях
- живёт на переводах → продукт для фрилансеров
- высокий остаток → накопительный продукт
- новый клиент → онбординг
- низкая активность → реактивация

Довнести конкретику по реальным профилям.

## 16. Демография по кластерам

Регион и пол не участвовали в кластеризации, но интересно посмотреть на состав после.

In [ ]:
demo_join = feature_table[['customer_id', 'cluster_kmeans']].merge(
    region_lookup.reset_index(), on='customer_id', how='left'
)

gender_breakdown = pd.crosstab(demo_join['cluster_kmeans'], demo_join['gender'], normalize='index').round(3) * 100
gender_breakdown.columns = [f'% {c}' for c in gender_breakdown.columns]
print('Пол по кластерам (%):')
print(gender_breakdown)


In [ ]:
region_breakdown = {}
for cl in sorted(demo_join['cluster_kmeans'].unique()):
    sub = demo_join[demo_join['cluster_kmeans'] == cl]
    top_regions = (sub['region'].value_counts(normalize=True) * 100).round(1).head(3)
    region_breakdown[cl] = ', '.join(f'{r} ({p}%)' for r, p in top_regions.items())

region_summary = pd.Series(region_breakdown, name='топ-3 региона').to_frame()
region_summary.index.name = 'cluster_kmeans'
region_summary
print(region_summary.to_string())

Если распределение внутри кластера близко к среднему по базе — демография не выделяет кластер. Если сильно отличается — дополнительный сигнал для таргетинга.

In [ ]:
overall_gender = (demo_join['gender'].value_counts(normalize=True) * 100).round(1)
overall_age = feature_table['age_years'].mean().round(1)
print('По всей базе: пол —', dict(overall_gender), '| средний возраст —', overall_age)
print()
age_by_cluster = feature_table.groupby('cluster_kmeans')['age_years'].mean().round(1).rename('средний возраст')
age_by_cluster.to_frame()
